# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [9]:
# Write your code below.
%load_ext dotenv
%dotenv

In [10]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [11]:
import os
from glob import glob

In [12]:
# Load envirnment variable 'PRICE_DATA'
PRICE_DATA = os.getenv("PRICE_DATA")

In [13]:
PRICE_DATA

'../../05_src/data/prices/'

In [14]:
# Use globe to find the path of all parquet files in the directory `PRICE_DATA`

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)

In [15]:
dd_px = dd.read_parquet(parquet_files)

In [19]:
dd_px.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382847,51.502148,56.464592,48.193848,56.330471,4674353.0,2000
A,2000-01-04 00:00:00+00:00,40.068874,47.567955,49.266811,46.316166,48.730328,4765083.0,2000
A,2000-01-05 00:00:00+00:00,37.583416,44.617310,47.567955,43.141991,47.389126,5758642.0,2000
A,2000-01-06 00:00:00+00:00,36.152367,42.918453,44.349072,41.577251,44.080830,2534434.0,2000
A,2000-01-07 00:00:00+00:00,39.165073,46.494991,47.165592,42.203148,42.247852,2819626.0,2000


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [20]:
# Add lags for variable 'Close'
dd_px["Close_lag_1"] = dd_px["Close"].shift(1)

In [23]:
# Add lags for variable 'Adj Clsoe'
dd_px["Adj_Close_lag_1"] = dd_px["Adj Close"].shift(1)

In [24]:
# Add 'returns' based on 'Close'
dd_px["returns"] = (dd_px["Close"] / dd_px["Close_lag_1"]) - 1

In [25]:
# Add 'Hi_lo_range'
dd_px["hi_lo_range"] = dd_px["High"] - dd_px["Low"]

In [26]:
# Asssign the resutls to dd_feat
dd_feat = dd_px[["Close", "Adj Close", "Close_lag_1", "Adj_Close_lag_1", "returns", "hi_lo_range"]]

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [27]:
# Convert Dask dataframe to pandas dataframe
df_pandas = dd_feat.compute()


In [28]:
df_pandas.head()

Price,Close,Adj Close,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,
A,51.502148,43.382847,NaN,NaN,NaN,8.270744
A,47.567955,40.068874,51.502148,43.382847,-0.076389,2.950645
A,44.617310,37.583416,47.567955,40.068874,-0.062030,4.425964
A,42.918453,36.152367,44.617310,37.583416,-0.038076,2.771820
A,46.494991,39.165073,42.918453,36.152367,0.083333,4.962444


In [30]:
# Add a new feature containing the moving average of returns using a window of 10 days
df_pandas["returns_mov_10"] = df_pandas["returns"].rolling(10).mean()
df_pandas.head()

Price,Close,Adj Close,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_mov_10
Ticker,,,,,,,
A,51.502148,43.382847,NaN,NaN,NaN,8.270744,NaN
A,47.567955,40.068874,51.502148,43.382847,-0.076389,2.950645,NaN
A,44.617310,37.583416,47.567955,40.068874,-0.062030,4.425964,NaN
A,42.918453,36.152367,44.617310,37.583416,-0.038076,2.771820,NaN
A,46.494991,39.165073,42.918453,36.152367,0.083333,4.962444,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
#It is necessary to convert dask data frame to pandas data frame as pandas handle .rolling function more effeicently. 
# Though I tried to use the .rolling function to get an appropraite value or moving returns but all the values turned out as NaN. 

# It can be done in Dask, but it is more complex. Since the given dataset is small, so to use pandas is a better and faster.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.